# Registro histórico: 12_oversampling_revision

Origem: `C29` — `Colab Notebooks/Untitled25.ipynb`.

Revisão com busca recursiva de imagens e uma saída binária.

**Limitações conhecidas:** Mantém outras inconsistências; a busca recursiva diferencia esta versão de C53. Não declarar resultado final sem reexecutar.

Cópia organizada em 25/10/2026. Código e texto originais preservados abaixo; saídas, contagens de execução e metadados do Colab removidos. As métricas salvas estão em `results/historical_evidence.json`. Não executar em lote: estes cadernos dependem do ambiente e de arquivos originais. Veja `docs/experiments.md` para o fluxo novo.


In [ ]:
# =============================================================================
# CÉLULA 1: INSTALAÇÃO E IMPORTS
# =============================================================================
import os
import random
import shutil
import zipfile
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights
from torch.optim.lr_scheduler import ReduceLROnPlateau

print("✅ Imports concluídos com sucesso!")

In [ ]:
# =============================================================================
# CÉLULA 2: CONFIGURAÇÕES INICIAIS E DEVICE
# =============================================================================
# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Usando device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# Set random seeds para reprodutibilidade
SEED = 91432
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"🌱 Random seed definida: {SEED}")


In [ ]:
# =============================================================================
# CÉLULA 3: DOWNLOAD DO DATASET HAM10000 DO KAGGLE
# =============================================================================
from google.colab import files

# Instalar kaggle
!pip install -q kaggle

# Upload do kaggle.json
print("📤 Faça upload do seu arquivo kaggle.json:")
print("   (Kaggle → Settings → API → Create New API Token)")
uploaded = files.upload()
!mv kaggle(1).json kaggle.json

# Configurar credenciais
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Baixar dataset
print("\n📦 Baixando dataset HAM10000 do Kaggle...")
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

# Extrair
print("\n📂 Extraindo arquivos...")
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/HAM10000

# Definir caminhos
BASE_DIR = Path('/content/HAM10000')
METADATA_FILE = BASE_DIR / 'HAM10000_metadata.csv'
IMAGE_DIR = BASE_DIR

# Verificar
print(f"\n✅ Dataset pronto!")
print(f"📄 Metadados: {METADATA_FILE}")
print(f"📂 Imagens em: {IMAGE_DIR}")
print(f"\n📋 Arquivos CSV disponíveis:")
!ls -lh /content/HAM10000/*.csv

In [ ]:
# =============================================================================
# CÉLULA 4: LOCALIZAR IMAGENS E PROCESSAR METADADOS
# =============================================================================
# Primeiro, vamos encontrar onde as imagens estão
print("🔍 Procurando imagens no diretório extraído...")
!find /content/HAM10000 -name "*.jpg" | head -5

# Listar estrutura de diretórios
print("\n📂 Estrutura de diretórios:")
!ls -R /content/HAM10000 | head -20

# Encontrar todas as imagens .jpg recursivamente
import glob
all_images = glob.glob(str(BASE_DIR / "**/*.jpg"), recursive=True)
print(f"\n✅ Total de imagens .jpg encontradas: {len(all_images)}")

if len(all_images) > 0:
    print(f"📍 Exemplo de caminho: {all_images[0]}")
    # Extrair o diretório base das imagens
    IMAGE_DIR = Path(all_images[0]).parent
    print(f"📂 Diretório das imagens: {IMAGE_DIR}")

# Carregar metadados
try:
    df = pd.read_csv(METADATA_FILE)
    print(f"\n✅ Metadados carregados: {len(df)} amostras")
except FileNotFoundError:
    print("❌ ERRO: Arquivo de metadados não encontrado!")
    raise

# Mapear para classificação binária
df['binary_label'] = df['dx'].apply(lambda x: 'melanoma' if x == 'mel' else 'non_melanoma')

# Análise inicial do desbalanceamento
print("\n📊 ANÁLISE DO DESBALANCEAMENTO:")
print("="*50)
class_counts = df['binary_label'].value_counts()
print(class_counts)
print(f"\nPercentual de Melanoma: {class_counts['melanoma']/len(df)*100:.2f}%")
print(f"Percentual de Não-Melanoma: {class_counts['non_melanoma']/len(df)*100:.2f}%")
print(f"Razão de desbalanceamento: 1:{class_counts['non_melanoma']/class_counts['melanoma']:.2f}")

# Criar um dicionário de image_id -> caminho completo
image_paths = {}
for img_path in all_images:
    img_name = Path(img_path).stem  # Nome sem extensão
    image_paths[img_name] = img_path

print(f"\n🗂️  Dicionário de imagens criado: {len(image_paths)} imagens mapeadas")

# Criar caminhos completos das imagens usando o dicionário
df['path'] = df['image_id'].apply(lambda x: image_paths.get(x, ''))

# Verificar existência dos arquivos
missing_images = ~df['path'].apply(lambda x: os.path.exists(x))
if missing_images.sum() > 0:
    print(f"\n⚠️  {missing_images.sum()} imagens não encontradas. Removendo do dataset...")
    df = df[~missing_images]

print(f"\n✅ Total de amostras válidas: {len(df)}")

# Divisão estratificada em treino/val/teste (80/10/10)
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['binary_label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['binary_label']
)

print("\n📂 DIVISÃO DOS DADOS:")
print("="*50)
print(f"Treino:    {len(train_df):5d} amostras ({len(train_df)/len(df)*100:.1f}%)")
print(f"Validação: {len(val_df):5d} amostras ({len(val_df)/len(df)*100:.1f}%)")
print(f"Teste:     {len(test_df):5d} amostras ({len(test_df)/len(df)*100:.1f}%)")

print("\n🔍 Distribuição por conjunto:")
for name, data in [("Treino", train_df), ("Validação", val_df), ("Teste", test_df)]:
    counts = data['binary_label'].value_counts()
    print(f"\n{name}:")
    print(f"  Melanoma:     {counts.get('melanoma', 0):4d}")
    print(f"  Não-Melanoma: {counts.get('non_melanoma', 0):4d}")


In [ ]:
# =============================================================================
# CÉLULA 5: CLASSES DE DATASET COM OVERSAMPLING
# =============================================================================
class OversampledImageDataset(Dataset):
    """Dataset com oversampling da classe minoritária por duplicação."""

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = {'non_melanoma': 0, 'melanoma': 1}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

        # Identificar classes
        counts = self.dataframe['binary_label'].value_counts()
        self.minority_class = counts.idxmin()
        self.majority_class = counts.idxmax()
        self.majority_count = counts.max()
        self.minority_count = counts.min()

        # Filtrar classe minoritária
        self.minority_df = self.dataframe[self.dataframe['binary_label'] == self.minority_class]

        # Calcular duplicação necessária
        self.duplication_factor = self.majority_count // self.minority_count
        self.remainder = self.majority_count % self.minority_count

        print(f"\n🔄 OVERSAMPLING CONFIGURADO:")
        print(f"   Classe minoritária: {self.minority_class} ({self.minority_count} amostras)")
        print(f"   Classe majoritária: {self.majority_class} ({self.majority_count} amostras)")
        print(f"   Fator de duplicação: {self.duplication_factor}x + {self.remainder} amostras")

        # Criar lista de índices com duplicações
        self.indices = list(self.dataframe.index)
        minority_indices = list(self.minority_df.index)

        # Duplicar
        for _ in range(self.duplication_factor):
            self.indices.extend(minority_indices)

        # Adicionar resto
        self.indices.extend(random.sample(minority_indices, self.remainder))

        # Dataset final
        self.final_df = self.dataframe.loc[self.indices].reset_index(drop=True)

        print(f"\n📊 RESULTADO DO OVERSAMPLING:")
        final_counts = self.final_df['binary_label'].value_counts()
        print(final_counts)
        print(f"   Total: {len(self.final_df)} amostras")

    def __len__(self):
        return len(self.final_df)

    def __getitem__(self, idx):
        row = self.final_df.iloc[idx]
        img_path = row['path']
        label_name = row['binary_label']
        label = self.class_to_idx[label_name]

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label


class SimpleImageDataset(Dataset):
    """Dataset simples sem oversampling (para validação/teste)."""

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = {'non_melanoma': 0, 'melanoma': 1}
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = row['path']
        label_name = row['binary_label']
        label = self.class_to_idx[label_name]

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# =============================================================================
# CÉLULA 6: DATA TRANSFORMS E DATALOADERS
# =============================================================================
# EfficientNet-B0 usa 224x224 com normalização ImageNet
# Mean e Std do ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transforms com data augmentation para treino
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Transforms simples para val/test
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Criar datasets
print("🔨 Criando datasets...")
train_dataset = OversampledImageDataset(train_df, transform=train_transform)
val_dataset = SimpleImageDataset(val_df, transform=test_transform)
test_dataset = SimpleImageDataset(test_df, transform=test_transform)

# Configurar DataLoaders
BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count() // 2 or 1

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if device == "cuda" else False
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if device == "cuda" else False
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if device == "cuda" else False
)

print(f"\n✅ DATALOADERS CRIADOS:")
print(f"   Treino:    {len(train_dataloader):4d} batches ({len(train_dataset)} amostras)")
print(f"   Validação: {len(val_dataloader):4d} batches ({len(val_dataset)} amostras)")
print(f"   Teste:     {len(test_dataloader):4d} batches ({len(test_dataset)} amostras)")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Workers: {NUM_WORKERS}")


In [ ]:
# =============================================================================
# CÉLULA 7: CRIAÇÃO DO MODELO EFFICIENTNET-B0
# =============================================================================
def create_efficientnet_b0_model(num_classes=1, pretrained=True):
    """Cria EfficientNet-B0 modificado para classificação binária."""

    if pretrained:
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        print("🔹 EfficientNet-B0 carregado com pesos do ImageNet")
    else:
        model = models.efficientnet_b0(weights=None)
        print("🔹 EfficientNet-B0 criado SEM pesos pré-treinados")

    # Congelar features iniciais (transfer learning)
    for param in model.features.parameters():
        param.requires_grad = False

    # Modificar classificador final
    num_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(num_features, num_classes)

    # Contar parâmetros
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"🔹 Classificador modificado para {num_classes} classe(s)")
    print(f"🔹 Total de parâmetros: {total_params:,}")
    print(f"🔹 Parâmetros treináveis: {trainable_params:,}")

    return model

# Criar modelo (1 saída para BCEWithLogitsLoss)
model = create_efficientnet_b0_model(num_classes=1, pretrained=True)
model = model.to(device)

print(f"\n✅ Modelo criado e movido para {device}")

In [ ]:
# =============================================================================
# CÉLULA 8: CONFIGURAÇÃO DO TREINAMENTO
# =============================================================================
# Calcular peso para classe positiva (Melanoma)
class_counts = train_df['binary_label'].value_counts()
pos_weight_value = class_counts['non_melanoma'] / class_counts['melanoma']
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float).to(device)

print(f"⚖️  CONFIGURAÇÃO DE PERDA:")
print(f"   Peso da classe positiva (Melanoma): {pos_weight_value:.2f}")
print(f"   Isso compensa o desbalanceamento {class_counts['non_melanoma']}:{class_counts['melanoma']}")

# Loss function
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Learning Rate Scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

print(f"\n🎯 CONFIGURAÇÕES DE TREINAMENTO:")
print(f"   Loss: BCEWithLogitsLoss (com peso {pos_weight_value:.2f})")
print(f"   Optimizer: AdamW (lr=1e-4, weight_decay=1e-4)")
print(f"   Scheduler: ReduceLROnPlateau (monitorando F1-Score)")

In [ ]:
# =============================================================================
# CÉLULA 9: FUNÇÕES DE MÉTRICA E TREINAMENTO
# =============================================================================
def calculate_metrics(y_true, y_pred_logits):
    """Calcula F1-Score e Acurácia."""
    y_pred_probs = torch.sigmoid(y_pred_logits)
    y_pred_labels = torch.round(y_pred_probs).squeeze()

    y_true_np = y_true.cpu().numpy()
    y_pred_labels_np = y_pred_labels.cpu().detach().numpy()

    f1 = f1_score(y_true_np, y_pred_labels_np, average='macro', zero_division=0)
    acc = (y_pred_labels == y_true).float().mean()

    return f1, acc


def train_step(model, data_loader, loss_fn, optimizer, device):
    """Passo de treinamento."""
    model.train()
    train_loss, train_f1, train_acc = 0, 0, 0

    progress_bar = tqdm(data_loader, desc="🔹 Training", leave=False)

    for X, y in progress_bar:
        X, y = X.to(device), y.to(device).float().unsqueeze(1)

        # Forward
        y_logits = model(X)
        loss = loss_fn(y_logits, y)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Métricas
        train_loss += loss.item()
        f1, acc = calculate_metrics(y, y_logits)
        train_f1 += f1
        train_acc += acc.item()

        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'f1': f'{f1:.4f}'
        })

    train_loss /= len(data_loader)
    train_f1 /= len(data_loader)
    train_acc /= len(data_loader)

    return train_loss, train_f1, train_acc


def val_step(model, data_loader, loss_fn, device):
    """Passo de validação."""
    model.eval()
    val_loss, val_f1, val_acc = 0, 0, 0
    all_preds, all_labels = [], []

    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device), y.to(device).float().unsqueeze(1)

            y_logits = model(X)
            loss = loss_fn(y_logits, y)
            val_loss += loss.item()

            f1, acc = calculate_metrics(y, y_logits)
            val_f1 += f1
            val_acc += acc.item()

            all_labels.extend(y.cpu().squeeze().tolist())
            all_preds.extend(torch.round(torch.sigmoid(y_logits)).cpu().squeeze().tolist())

    val_loss /= len(data_loader)
    val_f1 /= len(data_loader)
    val_acc /= len(data_loader)

    return val_loss, val_f1, val_acc, all_labels, all_preds


print("✅ Funções de treinamento e validação definidas")

In [ ]:
# =============================================================================
# CÉLULA 10: EARLY STOPPING
# =============================================================================
class F1EarlyStopping:
    """Early stopping baseado em F1-Score."""

    def __init__(self, patience=5, min_f1=0.75, verbose=False, delta=0):
        self.patience = patience
        self.min_f1 = min_f1
        self.verbose = verbose
        self.counter = 0
        self.best_f1 = -np.inf
        self.early_stop = False
        self.delta = delta
        self.best_model_state = None

    def __call__(self, val_f1, model):
        if val_f1 > self.best_f1 + self.delta:
            self.best_f1 = val_f1
            self.counter = 0
            self.best_model_state = model.state_dict()
            if self.verbose:
                print(f'   💾 F1-Score melhorou para {val_f1:.4f}')
        else:
            self.counter += 1
            if self.verbose:
                print(f'   ⏳ EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

        if val_f1 >= self.min_f1:
            print(f"   🎯 F1-Score alvo de {self.min_f1} atingido!")
            self.early_stop = True

        return self.early_stop


print("✅ Early Stopping configurado")

In [ ]:
# =============================================================================
# CÉLULA 11: LOOP DE TREINAMENTO PRINCIPAL
# =============================================================================
EPOCHS = 25
early_stopping = F1EarlyStopping(patience=5, min_f1=0.75)

train_losses, val_losses = [], []
train_f1s, val_f1s = [], []
train_accs, val_accs = [], []

print("\n" + "="*70)
print("🚀 INICIANDO TREINAMENTO")
print("="*70)

start_time = timer()

for epoch in range(EPOCHS):
    print(f"\n📍 Epoch {epoch+1}/{EPOCHS}")
    print("-" * 70)

    # Treino
    train_loss, train_f1, train_acc = train_step(
        model, train_dataloader, loss_fn, optimizer, device
    )
    train_losses.append(train_loss)
    train_f1s.append(train_f1)
    train_accs.append(train_acc)

    # Validação
    val_loss, val_f1, val_acc, _, _ = val_step(
        model, val_dataloader, loss_fn, device
    )
    val_losses.append(val_loss)
    val_f1s.append(val_f1)
    val_accs.append(val_acc)

    # Ajustar LR
    scheduler.step(val_f1)

    # Resultados
    print(f"   Train → Loss: {train_loss:.4f} | F1: {train_f1:.4f} | Acc: {train_acc*100:.2f}%")
    print(f"   Val   → Loss: {val_loss:.4f} | F1: {val_f1:.4f} | Acc: {val_acc*100:.2f}%")

    # Early Stopping
    if early_stopping(val_f1, model):
        print(f"\n⏹️  Early stopping na época {epoch + 1}")
        if early_stopping.best_model_state is not None:
            model.load_state_dict(early_stopping.best_model_state)
            print("   📥 Melhor modelo carregado")
        break

end_time = timer()
total_train_time = end_time - start_time

print("\n" + "="*70)
print(f"✅ TREINAMENTO CONCLUÍDO")
print(f"   Tempo total: {total_train_time:.2f}s ({total_train_time/60:.2f} min)")
print(f"   Épocas treinadas: {epoch + 1}/{EPOCHS}")
print(f"   Melhor F1-Score (Val): {early_stopping.best_f1:.4f}")
print("="*70)

In [ ]:
# =============================================================================
# CÉLULA 12: AVALIAÇÃO NO CONJUNTO DE TESTE
# =============================================================================
print("\n" + "="*70)
print("🧪 AVALIAÇÃO NO CONJUNTO DE TESTE")
print("="*70)

test_loss, test_f1, test_acc, test_labels, test_preds = val_step(
    model, test_dataloader, loss_fn, device
)

print(f"\n📊 RESULTADOS FINAIS:")
print(f"   Test Loss:     {test_loss:.4f}")
print(f"   Test F1-Score: {test_f1:.4f}")
print(f"   Test Accuracy: {test_acc*100:.2f}%")

# Relatório detalhado
class_names = ['non_melanoma', 'melanoma']
print("\n📈 RELATÓRIO DE CLASSIFICAÇÃO DETALHADO:")
print("="*70)
print(classification_report(test_labels, test_preds, target_names=class_names, zero_division=0))


In [ ]:
# =============================================================================
# CÉLULA 13: VISUALIZAÇÕES
# =============================================================================
# Matriz de Confusão
plt.figure(figsize=(10, 8))
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Contagem'})
plt.title('Matriz de Confusão - Conjunto de Teste', fontsize=16, fontweight='bold')
plt.ylabel('Classe Real', fontsize=12)
plt.xlabel('Classe Predita', fontsize=12)
plt.tight_layout()
plt.savefig("confusion_matrix_efficientnet.png", dpi=150, bbox_inches='tight')
plt.show()

# Histórico de Treinamento
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# F1-Score
axes[0].plot(train_f1s, label='Train F1', marker='o', linewidth=2)
axes[0].plot(val_f1s, label='Validation F1', marker='s', linewidth=2)
axes[0].set_title('F1-Score vs. Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epochs', fontsize=12)
axes[0].set_ylabel('F1-Score (Macro)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(train_losses, label='Train Loss', marker='o', linewidth=2)
axes[1].plot(val_losses, label='Validation Loss', marker='s', linewidth=2)
axes[1].set_title('Loss vs. Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epochs', fontsize=12)
axes[1].set_ylabel('Loss (BCEWithLogitsLoss)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_history_efficientnet.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualizações salvas!")

In [ ]:
# =============================================================================
# CÉLULA 14: SALVAMENTO DO MODELO E HISTÓRICO
# =============================================================================
MODEL_SAVE_PATH = Path("models")
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

MODEL_NAME = f"EfficientNetB0_Melanoma_F1_{test_f1:.4f}_E{epoch+1}.pth"
MODEL_FULL_PATH = MODEL_SAVE_PATH / MODEL_NAME

# Salvar modelo
torch.save(obj=model.state_dict(), f=MODEL_FULL_PATH)
print(f"💾 Modelo salvo em: {MODEL_FULL_PATH}")

# Salvar histórico
history = {
    "train_losses": train_losses,
    "val_losses": val_losses,
    "train_f1s": train_f1s,
    "val_f1s": val_f1s,
    "train_accs": train_accs,
    "val_accs": val_accs,
    "test_results": {
        "loss": test_loss,
        "f1_score": test_f1,
        "accuracy": test_acc
    },
    "training_time": total_train_time,
    "epochs_trained": epoch + 1,
    "best_val_f1": early_stopping.best_f1,
    "model_config": {
        "architecture": "EfficientNet-B0",
        "num_classes": 1,
        "pretrained": True,
        "batch_size": BATCH_SIZE,
        "learning_rate": 1e-4,
        "optimizer": "AdamW",
        "loss_function": "BCEWithLogitsLoss",
        "pos_weight": pos_weight_value,
        "seed": SEED
    }
}

import json
history_path = MODEL_SAVE_PATH / "training_history_efficientnet.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)

print(f"📊 Histórico salvo em: {history_path}")

print("\n" + "="*70)
print("🎉 PIPELINE COMPLETO!")
print("="*70)
print(f"✅ Modelo treinado e salvo")
print(f"✅ F1-Score no teste: {test_f1:.4f}")
print(f"✅ Acurácia no teste: {test_acc*100:.2f}%")
print(f"✅ Gráficos e matriz de confusão salvos")
print("="*70)